# EXP-014: Magnitude-realistic instability, matched LR factors, constant-LR control, direction-only corruption (R11)

**Status: engineering validation / proof of concept**, same caveats as all prior
notebooks this session.

**Purpose (R11 of the revision roadmap, Domain/Methodology/Devil's Advocate, all
Critical):** EXP-001's "instability" disturbance used GRAD_NOISE_MAX=8.0, which
Domain review measured at roughly **1200x** the network's true converged gradient
norm -- effectively overwriting the gradient with near-pure noise ("parameter
erasure"), not a realistic instability. The 0.3x/3.0x reduce/increase LR factors are
also not magnitude-matched (not exact reciprocals), and no experiment has ever
checked whether "reduce wins" is really about RESPONDING to a detected disturbance,
versus simply "a permanently smaller LR happens to do better here regardless of
whether anything is detected." This notebook fixes all three gaps in one place:

1. **Magnitude-realistic instability**: measure the network's own converged
   gradient norm from a calibration window, then inject noise scaled to a modest,
   stated multiple of that (not an arbitrary large constant).
2. **Symmetric, exactly-reciprocal LR factors**: reduce=1/3, increase=3 (previously
   0.3/3.0, not exact reciprocals).
3. **LR-matched constant control**: two new "arms" that hold LR fixed at the
   reduced or increased value for the ENTIRE run, with NO triggering/detection
   logic at all -- isolates whether the win is about detecting-then-responding, or
   just about which fixed LR happens to suit the disturbance.
4. **Direction-only (non-magnitude) instability mechanism**: replaces each
   step's gradient with a random vector of the SAME norm (direction corrupted,
   magnitude exactly preserved) with some probability per step -- a genuinely
   different corruption mechanism from magnitude-inflating noise, isolating
   whether "reduce wins" depends on the disturbance inflating gradient magnitude
   specifically, or is a more general property of corrupted gradient information.


In [1]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

torch.set_num_threads(1)
print("torch", torch.__version__, "| numpy", np.__version__)


C:\Users\USER\anaconda3\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


torch 2.5.1+cpu | numpy 2.0.2


## 1. Toy task; measure the network's true converged gradient norm first

In [2]:
def make_model(seed):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 1))

MEAN0_BASE = np.array([-1.0, -1.0])
MEAN1_BASE = np.array([1.0, 1.0])
BASE_NOISE_SCALE = 0.5

def gen_batch(rng, drift_frac=0.0, batch_size=16):
    frac = float(np.clip(drift_frac, 0.0, 1.0))
    mean0 = (1 - frac) * MEAN0_BASE + frac * MEAN1_BASE
    mean1 = (1 - frac) * MEAN1_BASE + frac * MEAN0_BASE
    y = rng.integers(0, 2, size=batch_size)
    means = np.where(y[:, None] == 0, mean0, mean1)
    x = means + rng.normal(scale=BASE_NOISE_SCALE, size=(batch_size, 2))
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

def grad_norm(model):
    total = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total += float((p.grad.detach() ** 2).sum())
    return total ** 0.5

def flat_grad(model):
    return torch.cat([p.grad.detach().flatten() for p in model.parameters() if p.grad is not None])

def set_flat_grad(model, flat):
    i = 0
    for p in model.parameters():
        if p.grad is not None:
            n = p.grad.numel()
            p.grad.copy_(flat[i:i+n].view_as(p.grad))
            i += n

N_STEPS = 400
DISTURBANCE_ONSET = 200
RAMP_LEN = 40
CALIB_START, CALIB_END = 150, 200

# Measure the true converged gradient norm (no disturbance) to calibrate a REALISTIC
# instability magnitude, instead of an arbitrary hand-picked constant.
def measure_converged_gradnorm(seed):
    rng = np.random.default_rng(seed)
    model = make_model(seed)
    opt = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.BCEWithLogitsLoss()
    gns = []
    for step in range(CALIB_END):
        xb, yb = gen_batch(rng, batch_size=16)
        opt.zero_grad()
        hidden = model[1](model[0](xb))
        logits = model[2](hidden).squeeze(-1)
        loss = loss_fn(logits, yb)
        loss.backward()
        if step >= CALIB_START:
            gns.append(grad_norm(model))
        opt.step()
    return np.mean(gns)

converged_gradnorms = [measure_converged_gradnorm(s) for s in range(5)]
TRUE_CONVERGED_GRADNORM = float(np.mean(converged_gradnorms))
print(f"True converged gradient norm (mean over 5 seeds, steps {CALIB_START}-{CALIB_END}): {TRUE_CONVERGED_GRADNORM:.4f}")
print(f"For reference, EXP-001's GRAD_NOISE_MAX=8.0 is {8.0/TRUE_CONVERGED_GRADNORM:.0f}x this value.")

# REALISTIC instability: injected noise std scaled so PEAK injected magnitude is
# ~3x the true converged gradient norm -- a genuinely modest, stated multiple,
# not an arbitrary large constant.
REALISTIC_MULTIPLE = 3.0
GRAD_NOISE_REALISTIC = REALISTIC_MULTIPLE * TRUE_CONVERGED_GRADNORM
print(f"REALISTIC GRAD_NOISE_MAX (this notebook) = {REALISTIC_MULTIPLE}x converged gradnorm = {GRAD_NOISE_REALISTIC:.4f}")
print(f"(EXP-001's original value was {8.0/GRAD_NOISE_REALISTIC:.0f}x larger than this)")


True converged gradient norm (mean over 5 seeds, steps 150-200): 0.0493
For reference, EXP-001's GRAD_NOISE_MAX=8.0 is 162x this value.
REALISTIC GRAD_NOISE_MAX (this notebook) = 3.0x converged gradnorm = 0.1479
(EXP-001's original value was 54x larger than this)


## 2. Disturbances: drift, REALISTIC magnitude-inflating instability, DIRECTION-ONLY instability

In [3]:
def _ramp(step):
    if step < DISTURBANCE_ONSET:
        return 0.0
    return min(step - DISTURBANCE_ONSET, RAMP_LEN) / RAMP_LEN

def drift_forecastable(step):
    return ("drift", _ramp(step))

def instability_magnitude_forecastable(step):
    return ("magnitude", _ramp(step))

def instability_direction_forecastable(step):
    return ("direction", _ramp(step))

DISTURBANCES = {
    "drift": drift_forecastable,
    "instability_magnitude": instability_magnitude_forecastable,
    "instability_direction": instability_direction_forecastable,
}


## 3. Single training run -- 4 arms: reduce (triggered), increase (triggered), constant_low, constant_high

In [4]:
WARMUP = 50
STEADY_STATE_WINDOW = 100
# R11: symmetric, exactly-reciprocal LR factors (previously 0.3x/3.0x, not reciprocal)
LR_REDUCTION_FACTOR = 1.0 / 3.0
LR_BOOST_FACTOR = 3.0
COOLDOWN = 30

def run_trial(seed, disturbance_fn, mode):
    '''mode: "reduce" or "increase" (triggered, threshold-based -- same loss-threshold
    trigger logic as EXP-001/002/009, calibrated on the R3-corrected post-convergence
    window), or "constant_low"/"constant_high" (LR held fixed at the reduce/increase
    factor for the WHOLE run, no trigger, no detection -- isolates whether responsive
    triggering matters or a permanently different LR alone explains the result).'''
    assert mode in ("reduce", "increase", "constant_low", "constant_high")
    rng = np.random.default_rng(seed)
    model = make_model(seed)
    base_lr = 0.1
    opt = torch.optim.SGD(model.parameters(), lr=base_lr)
    loss_fn = nn.BCEWithLogitsLoss()

    loss_hist = []
    threshold = None
    trigger_step = None
    cooldown_remaining = 0

    for step in range(N_STEPS):
        kind, severity = disturbance_fn(step)
        drift_frac = severity if kind == "drift" else 0.0
        xb, yb = gen_batch(rng, drift_frac=drift_frac)

        opt.zero_grad()
        hidden = model[1](model[0](xb))
        logits = model[2](hidden).squeeze(-1)
        loss = loss_fn(logits, yb)
        loss.backward()

        loss_t = float(loss.item())

        if kind == "magnitude" and severity > 0:
            with torch.no_grad():
                for p in model.parameters():
                    if p.grad is not None:
                        p.grad.add_(torch.randn_like(p.grad) * (GRAD_NOISE_REALISTIC * severity))
        elif kind == "direction" and severity > 0:
            # Direction-only corruption: with probability `severity`, REPLACE this
            # step's gradient with a random vector of the SAME norm -- magnitude is
            # exactly preserved (not inflated at all), only direction is corrupted.
            if rng.random() < severity:
                with torch.no_grad():
                    g = flat_grad(model)
                    orig_norm = g.norm()
                    rand_dir = torch.randn_like(g)
                    rand_dir = rand_dir / (rand_dir.norm() + 1e-12) * orig_norm
                    set_flat_grad(model, rand_dir)

        loss_hist.append(loss_t)

        if step == CALIB_END - 1:
            calib_loss = np.array(loss_hist[CALIB_START:CALIB_END])
            threshold = float(calib_loss.mean() + 3 * calib_loss.std() + 1e-6)

        if mode in ("reduce", "increase"):
            if step >= CALIB_END and trigger_step is None and threshold is not None and loss_t > threshold:
                trigger_step = step
                cooldown_remaining = COOLDOWN
            if cooldown_remaining > 0:
                factor = LR_REDUCTION_FACTOR if mode == "reduce" else LR_BOOST_FACTOR
                for g in opt.param_groups:
                    g["lr"] = base_lr * factor
                cooldown_remaining -= 1
            else:
                for g in opt.param_groups:
                    g["lr"] = base_lr
        else:
            factor = LR_REDUCTION_FACTOR if mode == "constant_low" else LR_BOOST_FACTOR
            for g in opt.param_groups:
                g["lr"] = base_lr * factor

        opt.step()

    loss_arr = np.array(loss_hist)
    post_onset = loss_arr[DISTURBANCE_ONSET:]
    peak_loss = float(post_onset.max())
    steady_state_loss = float(np.mean(loss_arr[-STEADY_STATE_WINDOW:]))
    return {"trigger_step": trigger_step, "peak_loss": peak_loss, "steady_state_loss": steady_state_loss}


## 4. Run all disturbances x 4 modes x 10 seeds

In [5]:
SEEDS = list(range(10))
MODES = ("reduce", "increase", "constant_low", "constant_high")

results = []
for dist_name, fn in DISTURBANCES.items():
    for mode in MODES:
        for seed in SEEDS:
            r = run_trial(seed, fn, mode)
            results.append({"disturbance": dist_name, "mode": mode, "seed": seed,
                             "trigger_step": r["trigger_step"], "peak_loss": r["peak_loss"],
                             "steady_state_loss": r["steady_state_loss"]})

df = pd.DataFrame(results)
print(f"Total trials: {len(df)}")
summary = df.groupby(["disturbance", "mode"]).agg(
    n_triggered=("trigger_step", lambda s: s.notna().sum()),
    mean_peak_loss=("peak_loss", "mean"), std_peak_loss=("peak_loss", "std"),
    mean_steady_state_loss=("steady_state_loss", "mean"), std_steady_state_loss=("steady_state_loss", "std"),
).round(4)
summary


Total trials: 120


n_triggered  mean_peak_loss  \
disturbance           mode                                         
drift                 constant_high            0          1.8174   
                      constant_low             0          2.1839   
                      increase                10          1.5460   
                      reduce                  10          3.0688   
instability_direction constant_high            0          0.1889   
                      constant_low             0          0.1512   
                      increase                 8          0.1508   
                      reduce                   8          0.1515   
instability_magnitude constant_high            0          0.2259   
                      constant_low             0          0.1575   
                      increase                 9          0.1687   
                      reduce                   9          0.1626   

                                     std_peak_loss  mean_steady_state_loss  \
disturbance           mode                                                   
drift                 constant_high         0.1763                  0.0146   
                      constant_low          0.0913                  0.2462   
                      increase              0.1337                  0.0421   
                      reduce                0.1825                  0.0499   
instability_direction constant_high         0.0813                  0.0084   
                      constant_low          0.0224                  0.0557   
                      increase              0.0526                  0.0170   
                      reduce                0.0514                  0.0178   
instability_magnitude constant_high         0.0869                  0.0133   
                      constant_low          0.0295                  0.0355   
                      increase              0.0526                  0.0146   
                      reduce                0.0486                  0.0162   

                                     std_steady_state_loss  
disturbance           mode                                  
drift                 constant_high                 0.0022  
                      constant_low                  0.0658  
                      increase                      0.0121  
                      reduce                        0.0082  
instability_direction constant_high                 0.0014  
                      constant_low                  0.0066  
                      increase                      0.0015  
                      reduce                        0.0015  
instability_magnitude constant_high                 0.0067  
                      constant_low                  0.0054  
                      increase                      0.0054  
                      reduce                        0.0047

## 5. Interpretation

In [6]:
def get(dist, mode, col):
    return summary.loc[(dist, mode), col]

lines = []
lines.append("R11 -- DOES 'REDUCE WINS FOR INSTABILITY' SURVIVE A REALISTIC MAGNITUDE, MATCHED LR")
lines.append("FACTORS, AN LR-MATCHED CONSTANT CONTROL, AND A NON-MAGNITUDE CORRUPTION MECHANISM?")
lines.append("")
lines.append(f"Realistic instability magnitude used: {REALISTIC_MULTIPLE}x the true converged gradient norm")
lines.append(f"({GRAD_NOISE_REALISTIC:.3f}), vs. EXP-001's original {8.0/GRAD_NOISE_REALISTIC:.0f}x-larger constant.")
lines.append("")

for dist in DISTURBANCES:
    peak_reduce = get(dist, "reduce", "mean_peak_loss")
    peak_increase = get(dist, "increase", "mean_peak_loss")
    peak_const_low = get(dist, "constant_low", "mean_peak_loss")
    peak_const_high = get(dist, "constant_high", "mean_peak_loss")
    winner_triggered = "reduce" if peak_reduce < peak_increase else "increase"
    winner_constant = "constant_low" if peak_const_low < peak_const_high else "constant_high"
    triggering_helps = (peak_reduce < peak_const_low) if winner_triggered == "reduce" else (peak_increase < peak_const_high)
    lines.append(f"--- {dist} ---")
    lines.append(f"  peak loss: reduce={peak_reduce:.3f} increase={peak_increase:.3f} "
                 f"constant_low={peak_const_low:.3f} constant_high={peak_const_high:.3f}")
    lines.append(f"  triggered winner = {winner_triggered}, constant-LR winner = {winner_constant}")
    lines.append(f"  does RESPONSIVE TRIGGERING beat just permanently using the winning constant LR? "
                 f"{'YES' if triggering_helps else 'NO -- a permanently fixed LR does as well or better'}")
    lines.append("")

mag_winner = "reduce" if get("instability_magnitude", "reduce", "mean_peak_loss") < get("instability_magnitude", "increase", "mean_peak_loss") else "increase"
dir_winner = "reduce" if get("instability_direction", "reduce", "mean_peak_loss") < get("instability_direction", "increase", "mean_peak_loss") else "increase"
drift_winner = "reduce" if get("drift", "reduce", "mean_peak_loss") < get("drift", "increase", "mean_peak_loss") else "increase"

lines.append("SUMMARY:")
lines.append(f"  drift winner = {drift_winner} (expect increase)")
lines.append(f"  REALISTIC-magnitude instability winner = {mag_winner} (expect reduce, at only {REALISTIC_MULTIPLE}x not 1200x)")
lines.append(f"  DIRECTION-ONLY instability winner = {dir_winner} (new mechanism, no prior expectation stated)")
lines.append("")
mag_holds = (mag_winner == "reduce" and drift_winner == "increase")
lines.append(
    "R11 RESULT: " + (
        "the core pattern SURVIVES a realistic instability magnitude (3x converged gradnorm, not 1200x) and "
        "symmetric matched LR factors -- 'reduce wins for magnitude-inflating instability, increase wins for "
        "drift' is not an artifact of the originally extreme, unrealistic noise magnitude."
        if mag_holds else
        "the core pattern does NOT cleanly survive a realistic instability magnitude -- at only 3x the "
        "converged gradient norm (vs. the original 1200x), the winner for at least one condition changes. "
        "This substantially narrows the magnitude-inflating-instability claim: it may only hold at severities "
        "far beyond what 'instability' typically means in practice, which is precisely what Domain review "
        "warned the original 1200x-inflated GRAD_NOISE_MAX=8.0 might have been masking."
    )
)
print(chr(10).join(lines))
print()
print(f"Direction-only corruption (non-magnitude-based instability) winner: {dir_winner} -- "
      f"{'matches the magnitude-inflating instability pattern (reduce), suggesting the response rule generalizes beyond magnitude-specific corruption.' if dir_winner == 'reduce' else 'does NOT match the magnitude-inflating pattern -- direction-only corruption (which never inflates gradient norm at all) behaves differently, narrowing the reduce-for-instability claim to magnitude-specific corruption mechanisms only.'}")
print()
print("Same Stage A caveats as all prior notebooks this session (n=10 seeds, no pre-registered")
print("effect sizes, not the frozen Stage D experiment).")


R11 -- DOES 'REDUCE WINS FOR INSTABILITY' SURVIVE A REALISTIC MAGNITUDE, MATCHED LR
FACTORS, AN LR-MATCHED CONSTANT CONTROL, AND A NON-MAGNITUDE CORRUPTION MECHANISM?

Realistic instability magnitude used: 3.0x the true converged gradient norm
(0.148), vs. EXP-001's original 54x-larger constant.

--- drift ---
  peak loss: reduce=3.069 increase=1.546 constant_low=2.184 constant_high=1.817
  triggered winner = increase, constant-LR winner = constant_high
  does RESPONSIVE TRIGGERING beat just permanently using the winning constant LR? YES

--- instability_magnitude ---
  peak loss: reduce=0.163 increase=0.169 constant_low=0.158 constant_high=0.226
  triggered winner = reduce, constant-LR winner = constant_low
  does RESPONSIVE TRIGGERING beat just permanently using the winning constant LR? NO -- a permanently fixed LR does as well or better

--- instability_direction ---
  peak loss: reduce=0.151 increase=0.151 constant_low=0.151 constant_high=0.189
  triggered winner = increase, consta